# KPMG 1A Intelligence Agent

In [2]:
from pypdf import PdfReader
from os import listdir
from os.path import isfile, join
#from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import PyPDFLoader, PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [3]:
DATA_PATH = "../data"

In [4]:
documents = PyPDFDirectoryLoader(path=DATA_PATH, mode="single").load()
print("Num Documents:", len(documents), ", Total Char Len:", sum([len(_.page_content) for _ in documents]))

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/AAAAAB+SFHello-Regular', '/Encoding': '/MacRomanEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(188, 0, 2375798603856), '/LastChar': 211, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [243, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 277, 451, 277, 0, 0, 456, 586, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 654, 632, 702, 703, 575, 551, 0, 722, 248, 0, 639, 547, 854, 0, 752, 608, 752, 627, 617, 608, 0, 654, 947, 0, 0, 0, 0, 0, 0, 0, 426, 0, 528, 585, 531, 585, 543, 334, 581, 568, 227, 227, 516, 229, 838, 557, 562, 581, 0, 346, 496, 331, 557, 514, 748, 499, 516, 0, 382, 0, 382, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 773, 0, 0, 0, 0, 0, 0, 0, 0, 476, 475]}, but is not installed. Consider install

Num Documents: 5 , Total Char Len: 374564


In [5]:
print(documents[0].page_content[:2000])

2026-08-21
Pandora’s AI Model Routing Box: Efficient
Allocation with Costly Value Estimation
Adam Fisch∗, Shubhendu Trivedi, Fantine Huot, William W. Cohen, Michael Kaisers, Mirella Lapata, Kate
Larson, Jacob Eisenstein∗
Google DeepMind
Heterogeneous AI systems composed of multiple models, architectures, harnesses, or inference-time
settings can improve quality and efficiency by routing queries to thespecialistwho can answer most
effectively at the lowest cost. Routing requires estimating each specialist’s expected return, but this value
estimation has a cost. Cheap estimators (e.g., embedding-based predictors) are fast but noisy, while ac-
curate estimators (e.g., fine-tuned models with access to retrieval results or partial reasoning traces) are
expensive. We formalize this tradeoff as an instance of Pandora’s Box, the classical problem of optimal
search with costly inspection. Under a Gaussian signal model, the resulting policies have closed-form
value-of-information expressions tha

In [9]:
print(documents[0].page_content[1800:2000])

n increase the strategic specialist’s utility at the expense of others.
1. Introduction
AI model providers now often offer heterogeneous model families spanning a wide range of costs and
capabilities:


In [10]:
chunks = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200).split_documents(documents)
print("number of chunks:", len(chunks), ", average chunk length:", sum([len(i.page_content) for i in chunks])/len(chunks))

number of chunks: 210 , average chunk length: 1941.5


In [11]:
print(chunks[0].page_content)

2026-08-21
Pandora’s AI Model Routing Box: Efficient
Allocation with Costly Value Estimation
Adam Fisch∗, Shubhendu Trivedi, Fantine Huot, William W. Cohen, Michael Kaisers, Mirella Lapata, Kate
Larson, Jacob Eisenstein∗
Google DeepMind
Heterogeneous AI systems composed of multiple models, architectures, harnesses, or inference-time
settings can improve quality and efficiency by routing queries to thespecialistwho can answer most
effectively at the lowest cost. Routing requires estimating each specialist’s expected return, but this value
estimation has a cost. Cheap estimators (e.g., embedding-based predictors) are fast but noisy, while ac-
curate estimators (e.g., fine-tuned models with access to retrieval results or partial reasoning traces) are
expensive. We formalize this tradeoff as an instance of Pandora’s Box, the classical problem of optimal
search with costly inspection. Under a Gaussian signal model, the resulting policies have closed-form
value-of-information expressions tha

In [12]:
print(chunks[1].page_content)

1. Introduction
AI model providers now often offer heterogeneous model families spanning a wide range of costs and
capabilities: smallandfastmodelsforsimpleprompts, largeandexpensiveonesforcomplextasks, and
augmented variants with tools, retrieval, or extended reasoning for more specialized tasks. This raises
the question of how to allocate each input to the model best suited for it, for a given level of cost sensi-
tivity. This is theroutingproblem, and a growing body of work addresses it by estimating each model’s
expected return on the input and selecting the maximizer (Feng et al., 2026; Hu et al., 2024; Shnitzer
et al., 2023). Butvalue estimationis neither free nor particularly easy. An embedding-based predictor
is cheap but noisy; a fine-tuned scoring model is more accurate but more expensive; and computing
partial solutions, executing tool calls, or running retrieval pipelines can be more expensive still. The
routing decision itself thus involves a cost-accuracy tradeoff, which 

In [13]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2161.79it/s]


In [14]:
retriever=vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":4})

In [15]:
test_query = "What are some AI benchmarks?"

scored_results = vectorstore.similarity_search_with_score(test_query, k=4)

print(f"Query: \"{test_query}\"")

print(f"\nTop 4 retrieved chunks (lower distance = more similar):\n")

for i, (doc, score) in enumerate(scored_results, 1):

    preview = doc.page_content[:200].replace("\n", " ")

    print(f"--- Chunk {i} (distance: {score:.4f}) ---")

    print(f"{preview}...\n")

Query: "What are some AI benchmarks?"

Top 4 retrieved chunks (lower distance = more similar):

--- Chunk 1 (distance: 0.6811) ---
algorithm it started from. What today’s agents do at the algorithmic link, then, is recover a competent default rather than design past one; whether that changes is the measurement this benchmark exis...

--- Chunk 2 (distance: 0.8664) ---
Mays, Benjamin Kinsella, Wyatt Thompson, et al. Paperbench: Evaluating ai’s ability to replicate ai research. arXiv preprint arXiv:2504.01848, 2025. [47] Xiangru Tang, Yuliang Liu, Zefan Cai, et al. M...

--- Chunk 3 (distance: 0.8766) ---
11 capacity, hyperparameters, and learning rules; accordingly, they find tuning and engineering easier than method invention. Evenautoresearch, which opens architecture, optimizer, and training-loop ...

--- Chunk 4 (distance: 0.8895) ---
score from0.094to0 .196, which is most of what effort buys. A machine learning scientist reads the training dynamics, names the mechanism that is fai